In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
df = pd.read_csv("../data/processed/jobs_merge.csv")

In [61]:
drop_columns = ["scraped_at", "source", "business_trip"]
clean_df = df.drop(columns=drop_columns)

In [62]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12640 entries, 0 to 12639
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   keyword            12640 non-null  str    
 1   job_id             12640 non-null  int64  
 2   title              12640 non-null  str    
 3   company            12640 non-null  str    
 4   province           12617 non-null  str    
 5   city               12640 non-null  str    
 6   salary_visible     12640 non-null  bool   
 7   internship         12037 non-null  object 
 8   remote             12037 non-null  object 
 9   urgent             12037 non-null  object 
 10  experience_years   12640 non-null  int64  
 11  description        12636 non-null  str    
 12  work_type          12636 non-null  str    
 13  seniority          12636 non-null  str    
 14  category           12636 non-null  str    
 15  company_size       12636 non-null  str    
 16  industries         12636 non-null

In [63]:
for col in ["internship", "remote", "urgent", "military"]:
    print(clean_df[col].value_counts(dropna=False))
    print("-"*50)

internship
False    11762
NaN        603
True       275
Name: count, dtype: int64
--------------------------------------------------
remote
False    11331
True       706
NaN        603
Name: count, dtype: int64
--------------------------------------------------
urgent
False    6079
True     5958
NaN       603
Name: count, dtype: int64
--------------------------------------------------
military
False    7316
True     5320
NaN         4
Name: count, dtype: int64
--------------------------------------------------


In [64]:
date_cols = ["first_activation", "activation", "expire"]
for col in date_cols:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")
clean_df[date_cols].dtypes

first_activation    datetime64[us, UTC]
activation          datetime64[us, UTC]
expire              datetime64[us, UTC]
dtype: object

In [65]:
for col in ["work_type", "seniority", "company_size", "gender", "province"]:
    clean_df[col] = clean_df[col].astype("string").str.strip()

In [66]:
def normalize_skill(skill):
    if pd.isna(skill):
        return skill
    
    skill = skill.strip()

    mapping = {
        "GIT": "Git",
        "Sql Server": "SQL Server",
        "MySql": "MySQL",
        "PostgreSql": "PostgreSQL",
        "Rest API": "REST API",
        "Html & CSS": "HTML & CSS",
        "Wordpress": "WordPress",
        "PowerBI": "Power BI",
        ".Net Core / .Net": ".NET Core / .NET",
        "Server Virtualization (ESX": "Server Virtualization"
    }

    return mapping.get(skill, skill)

In [67]:
def normalize_list(text, item_func=None):
    if pd.isna(text):
        return np.nan

    items = (item.strip() for item in text.split(","))

    if item_func is not None:
        items = (item_func(item) for item in items)

    return ",".join(sorted(set(items)))

In [68]:
columns = {"software": normalize_skill, "benefits": None, "industries": None}

for col, func in columns.items():
    clean_df[col] = clean_df[col].apply(lambda x: normalize_list(x, func))

In [70]:
job_df = clean_df.copy()
job_df = job_df.drop(columns=["salary_title"])
job_df = job_df.dropna(
    subset=[
        "description",
        "work_type",
        "seniority",
        "category",
        "company_size",
        "industries",
        "gender",
        "military",
        "work_days",
    ]
)

In [72]:
recommendation_columns = [
    "job_id",

    "title",
    "company",

    "province",
    "city",

    "category",
    "industries",

    "experience_years",
    "seniority",

    "software",
    "software_levels",

    "work_type",
    "remote",

    "company_size",

    "benefits",

    "salary_min",
    "salary_max",

    "required_age_min",
    "required_age_max",

    "gender",
    "military",

    "application_count",

    "description",
]
recommendation_df = job_df[recommendation_columns].copy()

In [76]:
def split_to_list(text):
    if pd.isna(text):
        return []

    return [item.strip() for item in text.split(",") if item.strip()]


multi_value_columns = ["software", "benefits", "industries", "category"]
for col in multi_value_columns:
    recommendation_df[col] = recommendation_df[col].apply(split_to_list)

In [78]:
cleaned_df = job_df.copy()

In [80]:
cleaned_df.to_csv("../data/processed/cleaned_jobs.csv", index=False, encoding="utf-8-sig")